In [ ]:
import sys 
import PIL
from PIL import Image
print(sys.executable)

In [ ]:
import vlm_stereo, os
print(vlm_stereo.__file__)
print(os.listdir(os.path.dirname(vlm_stereo.__file__)))

In [ ]:
import numpy as np
from vlm_stereo.stimuli import make_rds

# We are having it generate 100 dot patterns with different seeds. 
# Then checking how close to 0.5 the mean is and the standard deviation on the mean.

cov = [(make_rds(seed=s)== 0).mean() for s in range(100)]
print(np.mean(cov), np.std(cov))

In [ ]:
check = [(make_rds(seed=s)== 0).mean(axis=0) for s in range(10)]
print(np.shape(check))

In [ ]:
import numpy as np
from vlm_stereo.stimuli import make_rds

"""Now we are going to check the column and row means to make sure we do not have edge artifacts.
1. For each make_rds call, we will get the mean of how many values are equal to 0. This yields a (100, 448) array of coverage values, 
448 values for each of the 100 seeds.  
2. The outer np.mean call will give us the average across all runs, yielding one 448-long array of average col or row coverage values.
Result should be very close to 0.5, with no edge artifacts.""" 

cols = np.mean([(make_rds(seed=s, dot_size=8)== 0).mean(axis=0) for s in range(100, 200)], axis=0)
rows = np.mean([(make_rds(seed=s, dot_size=8)== 0).mean(axis=1) for s in range(100, 200)], axis=0) 
# For columns (axis=0) and rows(axis=1).
# Note that axis=0 for the outer np.mean call because the first mean creates a 1D array of row means.

# Checking the first and last 6 values of the column and row means to screen for edge artifacts. 
print(cols[:6].round(3), cols[-6:].round(3)) 
print(rows[:6].round(3), rows[-6:].round(3))



In [ ]:
# Now we will check it visually
import matplotlib.pyplot as plt

plt.plot(cols, label='columns', color = 'blue'); plt.plot(rows, label='rows', color = 'green'); 
plt.axhline(0.5, color='k', lw=0.5); plt.legend(); plt.ylim(0.47, 0.53); 
plt.title('Mean row & column coverage across 100 seeds');
plt.ylabel('Mean Coverage'); plt.xlabel('Column or Row Index'); 
plt.xticks(np.arange(0,449, step = 56));

In [ ]:
# Checking if the vectorized version of make_rds is equivalent to the non-vectorized version. 

import numpy as np
from vlm_stereo.stimuli import make_rds 
from vlm_stereo.stimuli import _random_positions
from vlm_stereo.stimuli import _stamp_loop
from vlm_stereo.stimuli import _stamp_vectorized

rng = np.random.default_rng(0)
pos = _random_positions(rng, (448, 420), 0.5, 4)
print(np.array_equal(_stamp_loop(pos, 4, (448, 420)),
                     _stamp_vectorized(pos, 4, (448, 420))))

print(np.array_equal(make_rds(seed=1), make_rds(seed=1)),
      np.array_equal(make_rds(seed=1), make_rds(seed=2)))


In [ ]:
import numpy as np
from PIL import Image
from vlm_stereo.stimuli import _random_positions, _in_region, _stamp_vectorized

rng = np.random.default_rng(0)
shape, dot_size, sq = (448, 448), 4, 224
top = left = (448 - sq) // 2

pos = _random_positions(rng, shape, 0.5, dot_size)
m = _in_region(pos, dot_size, top, left, sq, sq)
print(len(pos[m]) + len(pos[~m]) == len(pos))
Image.fromarray(_stamp_vectorized(pos[m], dot_size, shape))




In [ ]:
import numpy as np
from vlm_stereo.stimuli import make_rds

L, R = make_rds(seed = 0, disparity = 0)
print(np.array_equal(L,R))

In [ ]:
import numpy as np
from PIL import Image
from vlm_stereo.stimuli import make_rds

L, R = make_rds(seed = 0, sq_size = 100, disparity = -16)
gap = np.full((L.shape[0], 10), 128, dtype=np.uint8)
Image.fromarray(np.hstack([L, gap, R]))

In [ ]:
import numpy as np
from PIL import Image
from vlm_stereo.stimuli import make_rds

d, top, left, sq = 16, 112, 112, 224

L0, R0 = make_rds(seed=0, disparity=0)
print("d=0 identical:", np.array_equal(L0, R0))

L, R = make_rds(seed=0, disparity=d)
a = L[top+20:top+sq-20, left+40:left+sq-40]
b = R[top+20:top+sq-20, left+40-d:left+sq-40-d]
print("sign/magnitude:", np.array_equal(a, b))

# left eye's strip (left of square) vs. right eye's view of the same columns
print("strips differ:", not np.array_equal(L[top:top+sq, left:left+8],
                                           R[top:top+sq, left:left+8]))